In [ ]:
import os
import zipfile
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# 1. 경로 설정 및 압축 해제
BASE_PATH = '/home/sample'
ZIP_FILE_PATH = os.path.join(BASE_PATH, 'hugging-face-project.zip')
EXTRACT_DIR = os.path.join(BASE_PATH, 'extracted_project') # 해제될 폴더명

# 압축 해제 (최초 1회 실행)
if not os.path.exists(EXTRACT_DIR):
    print(f"압축 해제 시작: {ZIP_FILE_PATH}")
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("압축 해제 완료.")

# 2. 실제 데이터 경로 자동 매칭
# 압축 해제 시 폴더가 중첩될 수 있으므로 dataset 폴더를 직접 찾습니다.
try:
    target_root = list(Path(EXTRACT_DIR).rglob('dataset'))[0].parent
    PROJECT_ROOT = str(target_root)
except IndexError:
    print("Error: 'dataset' 폴더를 찾을 수 없습니다. 경로를 확인하세요.")
    PROJECT_ROOT = EXTRACT_DIR

TRAIN_SOURCE = os.path.join(PROJECT_ROOT, 'dataset', 'Training', '01.원천데이터')
TRAIN_LABEL  = os.path.join(PROJECT_ROOT, 'dataset', 'Training', '02.라벨링데이터')
VAL_SOURCE   = os.path.join(PROJECT_ROOT, 'dataset', 'Validation', '01.원천데이터')
VAL_LABEL    = os.path.join(PROJECT_ROOT, 'dataset', 'Validation', '02.라벨링데이터')

# 3. 데이터 로드 함수 (원본 로직 유지)
def load_patent_data(source_dir: str, label_dir: str, split_name: str) -> pd.DataFrame:
    records = []
    source_zips = sorted(Path(source_dir).glob('*.zip'))
    label_zips  = sorted(Path(label_dir).glob('*.zip'))

    # zip 파일명 매칭 키 추출 (TS_ 또는 TL_ 제거)
    def zip_key(p: Path) -> str:
        return '_'.join(p.stem.split('_')[1:])

    label_map = {zip_key(z): z for z in label_zips}

    for src_zip in tqdm(source_zips, desc=f'Loading {split_name}'):
        key = zip_key(src_zip)
        lbl_zip = label_map.get(key)
        if lbl_zip is None:
            print(f'[WARN] 라벨 없음: {src_zip.name}')
            continue

        # 라벨 읽기: {documentId -> Lno}
        label_lookup = {}
        with zipfile.ZipFile(lbl_zip, 'r') as lz:
            for fname in lz.namelist():
                if not fname.endswith('.json'): continue
                doc_id = Path(fname).name.replace('.json', '')
                try:
                    with lz.open(fname) as f:
                        obj = json.load(f)
                    label_lookup[doc_id] = obj['dataset']['Lno']
                except: pass

        # 원천 데이터 읽기 및 결합
        with zipfile.ZipFile(src_zip, 'r') as sz:
            for fname in sz.namelist():
                if not fname.endswith('.json'): continue
                doc_id = Path(fname).name.replace('.json', '')
                label = label_lookup.get(doc_id)
                if label is None: continue
                try:
                    with sz.open(fname) as f:
                        obj = json.load(f)
                    ds = obj['dataset']
                    title    = ds.get('invention_title', '') or ''
                    abstract = ds.get('abstract', '')        or ''
                    claims   = ds.get('claims', '')          or ''
                    text = f"{title} [SEP] {abstract} [SEP] {claims}".strip()
                    records.append({'doc_id': doc_id, 'text': text, 'label': label})
                except: pass

    return pd.DataFrame(records)

# 4. 실행 및 결과 확인
print(f"작업 경로: {PROJECT_ROOT}")
df_train   = load_patent_data(TRAIN_SOURCE, TRAIN_LABEL, 'Train')
df_valtest = load_patent_data(VAL_SOURCE,   VAL_LABEL,   'Validation')

print('\n=== 데이터 로드 완료 ===')
print(f'Train   : {len(df_train):,}건')
print(f'Val+Test: {len(df_valtest):,}건')

In [2]:
# ── Cell 4: 전처리 — 텍스트 정제 & 라벨 인코딩 ──────────────────────────────
import re
from sklearn.preprocessing import LabelEncoder

def clean_text(text: str, max_chars: int = 2000) -> str:
    """특수문자 정제 + 길이 제한 (DeBERTa max_length=512 토큰 기준 약 2000자)"""
    text = re.sub(r'\s+', ' ', text)             # 연속 공백 정리
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)  # 제어문자 제거
    return text[:max_chars].strip()

df_train['text']   = df_train['text'].apply(clean_text)
df_valtest['text'] = df_valtest['text'].apply(clean_text)

# 빈 텍스트 제거
df_train   = df_train[df_train['text'].str.len() > 10].reset_index(drop=True)
df_valtest = df_valtest[df_valtest['text'].str.len() > 10].reset_index(drop=True)

# 라벨 인코딩 (A=0, B=1, ..., K=10)
le = LabelEncoder()
le.fit(sorted(df_train['label'].unique()))
df_train['label_id']   = le.transform(df_train['label'])
df_valtest['label_id'] = le.transform(df_valtest['label'])

NUM_CLASSES = len(le.classes_)
print('클래스 수:', NUM_CLASSES)
print('클래스 목록:', list(le.classes_))
print('\n[텍스트 길이 통계 - Train]')
print(df_train['text'].str.len().describe())

클래스 수: 11
클래스 목록: [np.str_('A'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('J'), np.str_('K')]

[텍스트 길이 통계 - Train]
count    422591.000000
mean       1260.907565
std         527.701143
min          44.000000
25%         816.000000
50%        1194.000000
75%        1796.000000
max        2000.000000
Name: text, dtype: float64


In [3]:
# ── Cell 5: Train / Val / Test 분리 ─────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Validation 폴더 데이터를 val(80%) / test(20%) 로 분리 (stratify)
df_val, df_test = train_test_split(
    df_valtest,
    test_size=0.2,
    random_state=42,
    stratify=df_valtest['label_id']
)

df_val  = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print('=== Split 결과 ===')
print(f'Train : {len(df_train):,}건')
print(f'Val   : {len(df_val):,}건')
print(f'Test  : {len(df_test):,}건')

# 저장 (Drive에 캐시 — 다음 실행 때 재사용 가능)
CACHE_DIR = os.path.join(PROJECT_ROOT, 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

df_train.to_parquet(os.path.join(CACHE_DIR, 'train.parquet'), index=False)
df_val.to_parquet(os.path.join(CACHE_DIR,   'val.parquet'),   index=False)
df_test.to_parquet(os.path.join(CACHE_DIR,  'test.parquet'),  index=False)
print('\n✓ cache/ 에 parquet 저장 완료')

=== Split 결과 ===
Train : 422,591건
Val   : 40,896건
Test  : 10,224건

✓ cache/ 에 parquet 저장 완료


In [11]:
df_test

,doc_id,text,label,label_id
0,kr20170087047b1,브라운가스를 이용한 가스터빈 동력상승 방법 및 시스템 [SEP] 본 발명은 브라운가...,B,1
1,kr20190158883b1,황산 망간이 함유된 분말 식품 및 이의 제조 방법 [SEP] 본 발명의 일 실시 예...,G,6
2,kr20040067049b1,"운동수단을 갖는 의료용 침대 [SEP] 본 발명은 동력수단을 이용하여 환자의 상체,...",A,0
3,kr20190151699a,통신 상태에 따른 탑승자 서비스 제공 [SEP] 통신 상태에 따른 탑승자 서비스 제...,H,7
4,jp2014554676b2,건축물의 실내 마감재용친환경 수성도료 조성물 [SEP] (과제) 본 발명은 건축물의...,D,3
...,...,...,...,...
10219,kr20000003886b1,원자력발전소 가압기내부 검사로봇 [SEP] 본 발명은 원자력발전소 가압기내부 검사 ...,B,1
10220,kr20160009649a,HVDC 시스템에서의 제어장치 및 이의 동작방법 [SEP] 본 발명의 일 실시예에 ...,B,1
10221,kr20190080536b1,현장형 병원체 검출을 위한 생체시료 전처리 및 분자진단 올인원 키트 및 올인원 키트...,A,0
10222,kr20100087607a,"배기구 설치형 풍력 발전 장치 [SEP] 본 발명은 풍력 발전 장치에 있어서, 특히...",B,1


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

DEVICE = 'cuda' #if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

MODEL_NAME = 'LDKSolutions/KR-patent-deberta-large'
MAX_LEN    = 512
BATCH_SIZE = 8       # T4 기준 large 모델 → 8 안전. OOM 시 4로 줄이세요
EPOCHS     = 3
LR         = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PatentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df['text'].tolist()
        self.labels    = df['label_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_ds = PatentDataset(df_train, tokenizer, MAX_LEN)
val_ds   = PatentDataset(df_val,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

from transformers import AutoConfig

# 1. 모델 설정을 먼저 로드합니다.
config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = NUM_CLASSES
config.hidden_size = 1024 


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    config=config,
    ignore_mismatched_sizes=True 
).to(DEVICE)
#model = AutoModelForSequenceClassification.from_pretrained(
#    MODEL_NAME, num_labels=NUM_CLASSES
#).to(DEVICE)

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.06),
    num_training_steps=total_steps
)


def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE)
            )
            preds.extend(out.logits.argmax(-1).cpu().tolist())
            labels.extend(batch['label'].tolist())
    return accuracy_score(labels, preds), preds, labels


best_val_acc    = 0.0
MODEL_SAVE_PATH = os.path.join(PROJECT_ROOT, 'best_model')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['label'].to(DEVICE)
        )
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    val_acc, _, _ = evaluate(model, val_loader)
    print(f'Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(MODEL_SAVE_PATH)
        tokenizer.save_pretrained(MODEL_SAVE_PATH)
        print(f'  → 모델 저장 (val_acc={val_acc:.4f})')

print(f'\n학습 완료. Best Val Acc: {best_val_acc:.4f}')

Device: cuda


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: LDKSolutions/KR-patent-deberta-large
Key                                                              | Status     |                                                                                                 
-----------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------
cls.predictions.transform.LayerNorm.bias                         | UNEXPECTED |                                                                                                 
cls.predictions.transform.dense.bias                             | UNEXPECTED |                                                                                                 
cls.predictions.decoder.bias                                     | UNEXPECTED |                                                                                                 
cls.predictions.transform

Epoch 1/3:   0%|          | 0/52824 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
predict(model, df_test)

[1,
 6,
 0,
 5,
 3,
 2,
 0,
 2,
 3,
 4,
 5,
 3,
 3,
 5,
 0,
 0,
 1,
 0,
 0,
 1,
 2,
 7,
 1,
 2,
 0,
 9,
 2,
 2,
 0,
 7,
 2,
 2,
 5,
 6,
 4,
 0,
 4,
 6,
 3,
 5,
 1,
 8,
 1,
 5,
 1,
 0,
 6,
 6,
 0,
 6,
 0,
 6,
 7,
 10,
 6,
 6,
 2,
 5,
 3,
 0,
 0,
 0,
 3,
 5,
 10,
 8,
 0,
 8,
 6,
 10,
 1,
 3,
 9,
 2,
 3,
 4,
 1,
 4,
 2,
 6,
 10,
 2,
 3,
 8,
 5,
 2,
 3,
 10,
 2,
 1,
 3,
 0,
 6,
 1,
 8,
 2,
 0,
 6,
 7,
 5,
 8,
 1,
 2,
 1,
 7,
 1,
 2,
 5,
 4,
 0,
 4,
 10,
 6,
 0,
 1,
 6,
 0,
 0,
 4,
 3,
 4,
 5,
 3,
 5,
 1,
 3,
 6,
 1,
 0,
 5,
 2,
 5,
 4,
 3,
 5,
 0,
 6,
 3,
 5,
 5,
 4,
 2,
 2,
 2,
 0,
 9,
 3,
 10,
 3,
 1,
 10,
 8,
 5,
 1,
 2,
 4,
 3,
 0,
 6,
 4,
 3,
 6,
 1,
 2,
 0,
 5,
 1,
 7,
 1,
 1,
 1,
 4,
 2,
 5,
 5,
 0,
 7,
 1,
 4,
 7,
 6,
 2,
 8,
 0,
 2,
 0,
 0,
 0,
 2,
 1,
 5,
 9,
 3,
 6,
 5,
 0,
 0,
 6,
 2,
 7,
 2,
 0,
 0,
 3,
 4,
 2,
 1,
 4,
 0,
 6,
 0,
 4,
 7,
 5,
 2,
 1,
 2,
 6,
 1,
 5,
 4,
 6,
 2,
 8,
 2,
 0,
 9,
 4,
 1,
 0,
 0,
 3,
 2,
 0,
 7,
 1,
 5,
 6,
 4,
 8,
 4,
 0,
 2,
 0,
 0,
 6,
 3,
 3,
